# Step 1 — Data Collection (SteamSpy)

SteamSpy has two relevant endpoints:
- `/api.php?request=all&page=N` — bulk list of games (name, owners, playtime) but **no tags**
- `/api.php?request=appdetails&appid=XXX` — per-game details including **user-voted tags**

So we: (1) fetch the bulk list to get a ranked list of games, then (2) fetch per-game details for the top N to get tags.

**Rate limit**: SteamSpy allows ~4 req/sec. We use a 0.3s delay → 5000 games ≈ 25 minutes.

**Output**: `data/steamspy_games.csv`

In [38]:
import os
import json
import time
import requests
import pandas as pd

os.chdir(r"c:\Users\diogo\steam-recommender")
os.makedirs("data", exist_ok=True)
print(f"Working directory: {os.getcwd()}")

Working directory: c:\Users\diogo\steam-recommender


## Step 1a — Fetch bulk game list

Each page returns ~1000 games sorted by owner count. We grab 5 pages → top ~5000 games.

In [39]:
def parse_owners(owners_str: str) -> int:
    try:
        parts = owners_str.replace(",", "").split("..")
        return (int(parts[0].strip()) + int(parts[1].strip())) // 2
    except Exception:
        return 0

all_games = {}
for page in range(5):  # 5 pages = ~5000 games
    print(f"Fetching page {page}...")
    resp = requests.get(
        "https://steamspy.com/api.php",
        params={"request": "all", "page": page},
        timeout=30
    )
    resp.raise_for_status()
    all_games.update(resp.json())
    time.sleep(1)

print(f"Games in bulk list: {len(all_games)}")

Fetching page 0...
Fetching page 1...
Fetching page 2...
Fetching page 3...
Fetching page 4...
Games in bulk list: 4995


In [40]:
# Build a sorted DataFrame — we'll use this to pick which games to fetch details for
bulk_rows = []
for appid, g in all_games.items():
    if g.get("name", "").strip():
        bulk_rows.append({
            "appid": int(appid),
            "name": g["name"],
            "owners": parse_owners(g.get("owners", "0 .. 0")),
            "average_playtime": g.get("average_forever", 0),
            "ccu": g.get("ccu", 0),
        })

bulk_df = pd.DataFrame(bulk_rows).sort_values("owners", ascending=False).reset_index(drop=True)
print(f"Top 10 most-owned games:")
print(bulk_df[["name", "owners"]].head(10).to_string(index=False))

Top 10 most-owned games:
                            name    owners
Counter-Strike: Global Offensive 150000000
                    Apex Legends 150000000
             PUBG: BATTLEGROUNDS 150000000
                        Palworld  75000000
                 Team Fortress 2  75000000
 Call of Duty: Modern Warfare II  75000000
             New World: Aeternum  75000000
              Black Myth: Wukong  75000000
       Grand Theft Auto V Legacy  75000000
                   Left 4 Dead 2  75000000


## Step 1b — Fetch per-game details (with tags)

We fetch the top 5000 games by owner count. Each call returns the game's user-voted tags.

The cell saves progress every 100 games so you can resume if interrupted.

In [41]:
def fetch_appdetails(appid: int) -> dict | None:
    resp = requests.get(
        "https://steamspy.com/api.php",
        params={"request": "appdetails", "appid": appid},
        timeout=15
    )
    if resp.status_code != 200:
        return None
    d = resp.json()
    return {
        "appid": appid,
        "name": d.get("name", ""),
        "developer": d.get("developer", ""),
        "publisher": d.get("publisher", ""),
        "owners": parse_owners(d.get("owners", "0 .. 0")),
        "average_playtime": d.get("average_forever", 0),
        "median_playtime": d.get("median_forever", 0),
        "ccu": d.get("ccu", 0),
        "tags": json.dumps(d.get("tags") or {}),
    }

DETAILS_PATH = "data/steamspy_games.csv"
TOP_N = 5000

# Resume from existing file if interrupted
if os.path.exists(DETAILS_PATH):
    existing = pd.read_csv(DETAILS_PATH)
    done_ids = set(existing["appid"].tolist())
    print(f"Resuming — {len(done_ids)} games already fetched")
else:
    existing = pd.DataFrame()
    done_ids = set()

target_ids = [int(row.appid) for row in bulk_df.head(TOP_N).itertuples() if int(row.appid) not in done_ids]
print(f"Games left to fetch: {len(target_ids)}")

Resuming — 1000 games already fetched
Games left to fetch: 3995


In [42]:
results = []
for i, appid in enumerate(target_ids):
    detail = fetch_appdetails(appid)
    if detail:
        results.append(detail)

    if (i + 1) % 100 == 0:
        # Save progress
        batch = pd.DataFrame(results)
        combined = pd.concat([existing, batch], ignore_index=True) if not existing.empty else batch
        combined.to_csv(DETAILS_PATH, index=False)
        print(f"  {i + 1}/{len(target_ids)} fetched, saved {len(combined)} total")
        existing = combined
        results = []

    time.sleep(0.3)

# Save any remaining
if results:
    batch = pd.DataFrame(results)
    combined = pd.concat([existing, batch], ignore_index=True) if not existing.empty else batch
    combined.to_csv(DETAILS_PATH, index=False)
    existing = combined

print(f"\nDone! {len(existing)} games saved to {DETAILS_PATH}")

KeyboardInterrupt: 

## Inspect the results

In [ ]:
df = pd.read_csv(DETAILS_PATH)
df["tag_count"] = df["tags"].apply(lambda t: len(json.loads(t)))

print(f"Total games: {len(df)}")
print(f"Games with tags: {(df['tag_count'] > 0).sum()}")
print(f"Games without tags: {(df['tag_count'] == 0).sum()}")

print("\nSample (top 5 by owners):")
print(df.sort_values('owners', ascending=False)[["name", "owners", "tag_count"]].head())

Total games: 4995
Games with tags: 4992
Games without tags: 3

Sample (top 5 by owners):
                               name     owners  tag_count
0  Counter-Strike: Global Offensive  150000000         20
1                      Apex Legends  150000000         20
2               PUBG: BATTLEGROUNDS  150000000         20
3                          Palworld   75000000         20
4                   Team Fortress 2   75000000         20


In [ ]:
# Show tags for a known game
row = df[df["name"].str.lower().str.contains("Counter-Strike: Global Offensive")]
if not row.empty:
    print("Counter-Strike 2 tags:")
    print(json.loads(row.iloc[0]["tags"]))

In [ ]:
df1 = pd.read_csv("data/steamspy_games.csv")
df1

,Unnamed: 0,appid,name,developer,publisher,owners,average_playtime,median_playtime,ccu,tags
0,0,730,Counter-Strike: Global Offensive,Valve,Valve,150000000,0,0,1013936,"{""FPS"": 91172, ""Shooter"": 65634, ""Multiplayer""..."
1,1,1172470,Apex Legends,Respawn,Electronic Arts,150000000,0,0,124262,"{""Free to Play"": 2232, ""Battle Royale"": 1511, ..."
2,2,578080,PUBG: BATTLEGROUNDS,PUBG Corporation,"KRAFTON, Inc.",150000000,0,0,314682,"{""Survival"": 14893, ""Shooter"": 12788, ""Battle ..."
3,3,1623730,Palworld,Pocketpair,Pocketpair,75000000,0,0,18028,"{""Open World"": 1508, ""Survival"": 1382, ""Creatu..."
4,4,440,Team Fortress 2,Valve,Valve,75000000,0,0,43819,"{""Free to Play"": 62968, ""Hero Shooter"": 61037,..."
...,...,...,...,...,...,...,...,...,...,...
4990,4990,306660,Ultimate General: Gettysburg,Game-Labs,Game-Labs,350000,0,0,11,"{""Strategy"": 181, ""Historical"": 131, ""Simulati..."
4991,4991,335430,Grimoire: Manastorm,Omniconnection,Omniconnection,350000,0,0,1,"{""Action"": 53, ""Shooter"": 48, ""Indie"": 46, ""Ma..."
4992,4992,1462810,WRC 10 FIA World Rally Championship,KT Racing,Nacon,350000,0,0,44,"{""Racing"": 184, ""Automobile Sim"": 156, ""Sports..."
4993,4993,743390,DISTRAINT 2,Jesse Makkonen,Jesse Makkonen,350000,0,0,1,"{""Psychological Horror"": 188, ""Horror"": 180, ""..."


In [ ]:
df1.to_csv("data/steamspy_games.csv")

---
## Summary

- `data/steamspy_games.csv` — top 5000 games with user-voted tags

**Next**: `02_feature_engineering.ipynb`